# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [1]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.

---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [2]:
# batch_size와 epochs를 조정해보세요!
batch_size = 32
epochs = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# 작업 디렉터리 생성
!mkdir -p ~/data/multi30k

# train/validation/test 아카이브 다운로드
!wget --no-check-certificate -P ~/data/multi30k \
    https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/training.tar.gz \
    https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/validation.tar.gz \
    https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/mmt16_task1_test_2016.tar.gz

# 압축 풀기
!tar -xzf ~/data/multi30k/training.tar.gz   -C ~/data/multi30k
!tar -xzf ~/data/multi30k/validation.tar.gz -C ~/data/multi30k
!tar -xzf ~/data/multi30k/mmt16_task1_test_2016.tar.gz -C ~/data/multi30k

# 압축 해제 후 파일 확인
!ls -lh ~/data/multi30k

--2025-06-23 12:49:00--  https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/training.tar.gz
Resolving www.quest.dcs.shef.ac.uk (www.quest.dcs.shef.ac.uk)... 143.167.8.76
Connecting to www.quest.dcs.shef.ac.uk (www.quest.dcs.shef.ac.uk)|143.167.8.76|:443... connected.
  Issued certificate has expired.
	requested host name ‘www.quest.dcs.shef.ac.uk’.
HTTP request sent, awaiting response... 403 Forbidden
2025-06-23 12:49:01 ERROR 403: Forbidden.

--2025-06-23 12:49:01--  https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/validation.tar.gz
Reusing existing connection to www.quest.dcs.shef.ac.uk:443.
HTTP request sent, awaiting response... 403 Forbidden
2025-06-23 12:49:01 ERROR 403: Forbidden.

--2025-06-23 12:49:01--  https://www.quest.dcs.shef.ac.uk/wmt16_files_mmt/mmt16_task1_test_2016.tar.gz
Reusing existing connection to www.quest.dcs.shef.ac.uk:443.
HTTP request sent, awaiting response... 404 Not Found
2025-06-23 12:49:01 ERROR 404: Not Found.

tar (child): /root/data/multi30k/training.t

In [7]:
!rm -rf ./hf_cache
!pip install fsspec==2023.6.0 -q

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw_datasets = load_dataset("imdb", cache_dir="./hf_cache")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenized_datasets = raw_datasets.map(
    lambda x: tokenizer(x["text"], truncation=True, padding="max_length"),
    batched=True,
    load_from_cache_file=False
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
# 데이터셋 로드
raw_datasets

DatasetDict({
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [7]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [10]:
# 학습 함수 정의
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=32)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(4):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc


# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

KeyError: 'validation'

## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명
- BERT
  -  	Transformer 기반 양방향 인코더
  - pretraining 방식 : Masked Language Modeling (MLM) - 일부 토큰을 [MASK]로 가려 예측
  - 토큰을 가리는 방식이 인위적이며, 학습 비효율 가능
  - 상대적으로 학습 느림 (MLM 방식의 한계)

- ELECTRA
  - Generator + Discriminator 구조
  - pretraining 방식 : Replaced Token Detection (RTD) - Generator가 바꾼 토큰을 Discriminator가 식별
  - 모든 토큰에 대해 학습 → 더 빠르고 효과적인 pretraining
  - 빠르고 가볍지만 강력한 성능을 냄

2. 어떤 모델이 적합한지에 대한 본인의 의견   
  → 전반적으로 ELECTRA가 더 적합하다고 판단됨

- 학습 속도
  
  ELECTRA는 RTD 방식 덕분에 같은 데이터셋에서 더 빠른 수렴을 보이며, 에폭당 소요 시간도 짧음
  
  → Colab과 같은 리소스 제한 환경에 유리

- 정확도 (Accuracy)
  
  실험 결과에서도 ELECTRA가 BERT와 비슷하거나 더 높은 정확도를 보임
  
  → 적은 연산량 대비 높은 성능

- 모델 크기와 자원 효율성
  
  ELECTRA는 BERT 대비 더 적은 파라미터 수로 유사 성능을 내므로, 모바일/웹 애플리케이션 배포 관점에서도 유리



-
-